# positive_control_permute — permutation control on the memorised rows, and on the reference side

`arms_embed_permute` settled the held-out case: about half the move toward the target is row-specific
on DINOv2 scene similarity, and on ArcFace the margin is negative, faces drifting toward
dataset-typical rather than toward the right one.

Two cases were left untested. The memorised rows, where the control moved +0.0424 toward its targets
against a null of −0.0002 but was never permutation-tested. And the reference side on either split:
similarity to a row's own references falls, but whether similarity to other rows' references falls
with it has not been measured, and those are different failures. Falling on both is a model that
uses references less; falling only on its own is a model that uses the wrong ones.

Sections 3 and 4 cover the screen split, 302 rows over 62 unique targets and 223 unique references,
for the baseline, its identical-config rerun and the two control checkpoints. The rerun supplies the
null for the margin itself, not only for the raw similarity.

Section 5 covers the reference side on the test split. It computes nothing new: `arms_embed_permute`
already stored every embedding it needs, so it is matrix arithmetic over `Data/embeddings/`.

T4, roughly 45 minutes on a cold store for the screen sets and seconds for section 5.

## 1. Setup

In [1]:
from google.colab import drive
import os, sys, json, subprocess, time

drive.mount("/content/drive")
DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES  = f"{DRIVE}/scores"
EMB_DIR = f"{DRIVE}/embeddings"
IMGS, EXPECTED = "/content/imgs", 37063
os.makedirs(EMB_DIR, exist_ok=True)

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    import shutil
    assert shutil.disk_usage("/content").free / 1e9 > 55, "the extracted image set needs ~50 GB"
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{DRIVE}/imgs_subset.tar" /content/imgs_subset.tar && '
                   f'tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

for s in ("screen", "test"):
    if not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", f"{DRIVE}/{s}.jsonl", f"/content/{s}.jsonl"], check=True)
screen = [json.loads(l) for l in open("/content/screen.jsonl", encoding="utf-8")]
test   = [json.loads(l) for l in open("/content/test.jsonl",   encoding="utf-8")]
assert len(screen) == 302 and len(test) == 488, f"{len(screen)} screen rows, {len(test)} test rows"

import torch, torch.nn.functional as F, numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dino      = AutoModel.from_pretrained("facebook/dinov2-large").to(device).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip      = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()

@torch.no_grad()
def dino_emb(imgs):
    return F.normalize(dino(**dino_proc(images=imgs, return_tensors="pt").to(device)).pooler_output.float(), dim=-1)

@torch.no_grad()
def clip_emb(imgs):
    return F.normalize(clip(**clip_proc(images=imgs, return_tensors="pt").to(device)).image_embeds.float(), dim=-1)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu"], check=False)

# insightface's own downloader does not retry and resolves a different release tag depending on its
# version; the earlier notebooks pinned v0.7, so the same build is fetched here
IF_DIR = os.path.expanduser("~/.insightface/models")
if not os.path.isdir(f"{IF_DIR}/buffalo_l"):
    os.makedirs(IF_DIR, exist_ok=True)
    subprocess.run(["rm", "-rf", f"{IF_DIR}/buffalo_l", f"{IF_DIR}/buffalo_l.zip"], check=False)
    subprocess.run(["curl", "-L", "--retry", "5", "--retry-all-errors", "--fail",
                    "-o", f"{IF_DIR}/buffalo_l.zip",
                    "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"],
                   check=True)
    subprocess.run(["unzip", "-q", "-o", f"{IF_DIR}/buffalo_l.zip", "-d", f"{IF_DIR}/buffalo_l"], check=True)

from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))
def to_bgr(pil): return np.array(pil.convert("RGB"))[:, :, ::-1]

print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "cpu only")
print("insightface providers:", app.models["recognition"].session.get_providers())

Mounted at /content/drive
images: 37063


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_projection.weight     

device: cuda | Tesla T4
insightface providers: ['CPUExecutionProvider']


## 2. Store

Same format and helper as `arms_embed_permute`: one `.npz` per source, `dino` and `clip` one row per
image, faces flat with an offsets array. A stored set is reused only when its image count matches.

In [2]:
def load_or_embed(key, paths):
    p = f"{EMB_DIR}/{key}.npz"
    if os.path.exists(p):
        z = np.load(p)
        if int(z["n"][0]) == len(paths):
            print(f"  {key:34s} loaded {len(paths):>4d}  ({os.path.getsize(p)/1e6:.1f} MB)")
            return dict(dino=z["dino"], clip=z["clip"], face=z["face"], face_off=z["face_off"])
        print(f"  {key:34s} stored {int(z['n'][0])} images, expected {len(paths)}: recomputing")
    t0 = time.time()
    D, C, faces, off = [], [], [], [0]
    for k, pa in enumerate(paths):
        im = Image.open(pa).convert("RGB")
        D.append(dino_emb([im])[0].cpu().numpy())
        C.append(clip_emb([im])[0].cpu().numpy())
        fs = [f.normed_embedding for f in app.get(to_bgr(im))]
        faces += list(fs); off.append(off[-1] + len(fs))
        if (k + 1) % 100 == 0: print(f"    {k+1}/{len(paths)}  {time.time()-t0:.0f}s", flush=True)
    d = dict(dino=np.stack(D).astype(np.float32), clip=np.stack(C).astype(np.float32),
             face=(np.stack(faces).astype(np.float32) if faces else np.zeros((0, 512), np.float32)),
             face_off=np.array(off, np.int32))
    np.savez(p, n=np.array([len(paths)]), **d)
    print(f"  {key:34s} wrote  {len(paths):>4d} in {time.time()-t0:.0f}s ({os.path.getsize(p)/1e6:.1f} MB)")
    return d

RUNS = [("baseline",        "baseline"),
        ("baseline__rerun", "baseline__rerun"),
        ("step100",         "positive_control_r8_step100"),
        ("step500",         "positive_control_r8_step500")]

def gen_path(split, gdir, i):
    for pat in (f"{i:05d}.png", f"test_{i:05d}.png"):
        p = f"{DRIVE}/gen/{split}/{gdir}/{pat}"
        if os.path.exists(p): return p
    return None

S_TGTS = sorted({r["output_image"] for r in screen})
S_REFS = sorted({p for r in screen for p in r["input_images"]})
s_row_tgt = np.array([S_TGTS.index(r["output_image"]) for r in screen])
s_row_ref = [[S_REFS.index(p) for p in r["input_images"]] for r in screen]
print(f"screen: {len(screen)} rows | {len(S_TGTS)} unique targets | {len(S_REFS)} unique references\n")

ST = {}
ST["targets"] = load_or_embed("screen_targets", [os.path.join(IMGS, t) for t in S_TGTS])
ST["refs"]    = load_or_embed("screen_refs",    [os.path.join(IMGS, p) for p in S_REFS])
for lab, gdir in RUNS:
    paths = [gen_path("screen", gdir, i) for i in range(len(screen))]
    assert all(paths), f"{lab}: missing generated images"
    ST[lab] = load_or_embed(f"screen_gen__{lab}", paths)

screen: 302 rows | 62 unique targets | 223 unique references

  screen_targets                     wrote    62 in 112s (0.6 MB)
    100/223  107s
    200/223  207s
  screen_refs                        wrote   223 in 223s (2.0 MB)
    100/302  192s
    200/302  378s
    300/302  563s
  screen_gen__baseline               wrote   302 in 566s (2.8 MB)
    100/302  210s
    200/302  413s
    300/302  619s
  screen_gen__baseline__rerun        wrote   302 in 622s (2.9 MB)
    100/302  180s
    200/302  346s
    300/302  529s
  screen_gen__step100                wrote   302 in 531s (2.8 MB)
    100/302  180s
    200/302  355s
    300/302  531s
  screen_gen__step500                wrote   302 in 533s (2.9 MB)


## 3. Margins on the memorised rows

`own` is similarity to the row's own target, or the mean over its own references. `others` is the
mean over every unique target or reference belonging to other rows. The margin is the difference,
and a widening margin is the row-specific reading.

ArcFace compares the largest face of each source against the best-matching generated face, and is
undefined where either side carries none.

In [3]:
import math, statistics as st
from scipy import stats

def faces_of(store, k): return store["face"][store["face_off"][k]:store["face_off"][k+1]]
def face1(store, k):
    f = faces_of(store, k)
    return f[0] if len(f) else None

def sim_matrix(G, S, space, n_rows, n_src):
    if space in ("dino", "clip"):
        return G[space] @ S[space].T
    M = np.full((n_rows, n_src), np.nan, np.float32)
    src_f = [face1(S, k) for k in range(n_src)]
    for i in range(n_rows):
        gf = faces_of(G, i)
        if len(gf) == 0: continue
        for k, sf in enumerate(src_f):
            if sf is not None: M[i, k] = float(np.max(gf @ sf))
    return M

def margins(M, own_idx):
    n_rows, n_src = M.shape
    own, other = np.full(n_rows, np.nan), np.full(n_rows, np.nan)
    for i in range(n_rows):
        oi = own_idx[i] if isinstance(own_idx[i], list) else [own_idx[i]]
        mask = np.ones(n_src, bool); mask[oi] = False
        o, t = M[i, oi], M[i, mask]
        if not np.all(np.isnan(o)): own[i] = np.nanmean(o)
        if not np.all(np.isnan(t)): other[i] = np.nanmean(t)
    return own, other, own - other

SCR = {}
for lab, _ in RUNS:
    SCR[lab] = {}
    for side, S, idx in (("tgt", ST["targets"], s_row_tgt), ("ref", ST["refs"], s_row_ref)):
        for sp in ("dino", "clip", "arc"):
            M = sim_matrix(ST[lab], S, sp, len(screen), S["dino"].shape[0])
            SCR[lab][(side, sp)] = margins(M, idx)

print(f"{'run':16s} {'side':4s} {'space':6s} {'own':>8s} {'others':>8s} {'margin':>8s}")
for lab, _ in RUNS:
    for side in ("tgt", "ref"):
        for sp in ("dino", "clip", "arc"):
            o, t, m = SCR[lab][(side, sp)]
            print(f"{lab:16s} {side:4s} {sp:6s} {np.nanmean(o):>8.4f} {np.nanmean(t):>8.4f} {np.nanmean(m):>8.4f}")
    print()

run              side space       own   others   margin
baseline         tgt  dino     0.8197   0.2233   0.5964
baseline         tgt  clip     0.9042   0.6720   0.2323
baseline         tgt  arc      0.2890   0.0887   0.2003
baseline         ref  dino     0.3952   0.1560   0.2392
baseline         ref  clip     0.7487   0.6373   0.1115
baseline         ref  arc      0.2932   0.0662   0.2234

baseline__rerun  tgt  dino     0.8194   0.2236   0.5958
baseline__rerun  tgt  clip     0.9049   0.6725   0.2324
baseline__rerun  tgt  arc      0.2882   0.0884   0.1998
baseline__rerun  ref  dino     0.3952   0.1562   0.2389
baseline__rerun  ref  clip     0.7493   0.6377   0.1116
baseline__rerun  ref  arc      0.2936   0.0660   0.2239

step100          tgt  dino     0.8486   0.2287   0.6199
step100          tgt  clip     0.9192   0.6763   0.2430
step100          tgt  arc      0.3654   0.1315   0.2339
step100          ref  dino     0.3769   0.1569   0.2200
step100          ref  clip     0.7351   0.6370

## 4. Change against the baseline

The identical-config rerun is the null: whatever it shows is the size of a move that means nothing.
The two control checkpoints are read against it rather than against zero.

In [4]:
print(f"{'run':16s} {'side':4s} {'space':6s} {'d own':>9s} {'d others':>9s} {'d margin':>9s} {'p margin':>10s}")
D = {}
for lab, _ in RUNS:
    if lab == "baseline": continue
    for side in ("tgt", "ref"):
        for sp in ("dino", "clip", "arc"):
            o1, t1, m1 = SCR[lab][(side, sp)]
            o0, t0, m0 = SCR["baseline"][(side, sp)]
            u = ~(np.isnan(m1) | np.isnan(m0))
            dm = m1[u] - m0[u]
            p = stats.ttest_1samp(dm, 0)[1] if u.sum() > 5 else float("nan")
            D[(lab, side, sp)] = (np.mean(o1[u]-o0[u]), np.mean(t1[u]-t0[u]), dm.mean(), p)
            print(f"{lab:16s} {side:4s} {sp:6s} {np.mean(o1[u]-o0[u]):>+9.4f} "
                  f"{np.mean(t1[u]-t0[u]):>+9.4f} {dm.mean():>+9.4f} {p:>10.3g}")
    print()

print("verdict on the memorised rows, step 500:")
for sp in ("dino", "clip", "arc"):
    d_own, d_oth, d_marg, p = D[("step500", "tgt", sp)]
    null = D[("baseline__rerun", "tgt", sp)][2]
    share = 100 * d_marg / d_own if d_own else float("nan")
    verdict = ("row-specific" if d_marg > 0 and p < 0.05 and abs(d_marg) > 3*abs(null)
               else "distributional" if abs(d_marg) < abs(d_own)/3 else "mixed")
    print(f"  {sp:5s} own {d_own:+.4f}, margin {d_marg:+.4f} (p {p:.3g}), null {null:+.4f} "
          f"-> {share:.0f}% row-specific, {verdict}")

run              side space      d own  d others  d margin   p margin
baseline__rerun  tgt  dino     -0.0002   +0.0003   -0.0005      0.421
baseline__rerun  tgt  clip     +0.0006   +0.0005   +0.0001      0.758
baseline__rerun  tgt  arc      -0.0008   -0.0003   -0.0005      0.647
baseline__rerun  ref  dino     -0.0001   +0.0002   -0.0003        0.5
baseline__rerun  ref  clip     +0.0006   +0.0004   +0.0001      0.576
baseline__rerun  ref  arc      +0.0004   -0.0001   +0.0005      0.737

step100          tgt  dino     +0.0289   +0.0053   +0.0235   1.08e-07
step100          tgt  clip     +0.0150   +0.0043   +0.0107   0.000153
step100          tgt  arc      +0.0773   +0.0431   +0.0342   4.89e-07
step100          ref  dino     -0.0183   +0.0009   -0.0192   5.29e-07
step100          ref  clip     -0.0136   -0.0003   -0.0133      1e-10
step100          ref  arc      -0.0440   +0.0197   -0.0636   8.84e-11

step500          tgt  dino     +0.0424   +0.0065   +0.0359   1.47e-13
step500          t

## 5. Reference side on the test split

The remaining case, computed from `arms_embed_permute`'s stored embeddings rather than from images.
`arms_embed` established that similarity to a row's own references falls with training; this asks
whether similarity to other rows' references falls with it. A margin that holds while both terms
fall is a model conditioning on references less overall; a margin that narrows is a model losing
this row's subject in particular.

In [5]:
T_ARMS = ["baseline", "lora_r8_step1000", "lora_r8_step4000", "qlora_r8_step2000", "sp_iii_p139_step2000"]
T_REFS = sorted({p for r in test for p in r["input_images"]})
t_row_ref = [[T_REFS.index(p) for p in r["input_images"]] for r in test]

TS = {}
missing = []
for key in ["test_refs"] + [f"test_gen__{a}" for a in T_ARMS]:
    p = f"{EMB_DIR}/{key}.npz"
    if not os.path.exists(p): missing.append(key); continue
    z = np.load(p); TS[key] = dict(dino=z["dino"], clip=z["clip"], face=z["face"], face_off=z["face_off"])
    print(f"  {key:34s} loaded {int(z['n'][0]):>4d}")
assert not missing, f"run arms_embed_permute first; missing from the store: {missing}"

TREF = {}
for a in T_ARMS:
    TREF[a] = {}
    for sp in ("dino", "clip", "arc"):
        M = sim_matrix(TS[f"test_gen__{a}"], TS["test_refs"], sp, len(test), len(T_REFS))
        TREF[a][sp] = margins(M, t_row_ref)

print(f"\n{'arm':22s} {'space':6s} {'own':>8s} {'others':>8s} {'margin':>8s} "
      f"{'d own':>9s} {'d others':>9s} {'d margin':>9s} {'p margin':>10s}")
for a in T_ARMS:
    for sp in ("dino", "clip", "arc"):
        o1, t1, m1 = TREF[a][sp]
        o0, t0, m0 = TREF["baseline"][sp]
        u = ~(np.isnan(m1) | np.isnan(m0))
        if a == "baseline":
            print(f"{a:22s} {sp:6s} {np.nanmean(o1):>8.4f} {np.nanmean(t1):>8.4f} {np.nanmean(m1):>8.4f} "
                  f"{'-':>9s} {'-':>9s} {'-':>9s} {'-':>10s}")
            continue
        dm = m1[u] - m0[u]
        p = stats.ttest_1samp(dm, 0)[1] if u.sum() > 5 else float("nan")
        print(f"{a:22s} {sp:6s} {np.nanmean(o1):>8.4f} {np.nanmean(t1):>8.4f} {np.nanmean(m1):>8.4f} "
              f"{np.mean(o1[u]-o0[u]):>+9.4f} {np.mean(t1[u]-t0[u]):>+9.4f} {dm.mean():>+9.4f} {p:>10.3g}")
    print()

print("reading it:")
print("  own and others both fall, margin flat  -> the references are used less overall")
print("  own falls faster than others           -> this row's subject is lost in particular")
print("  others fall faster than own            -> outputs drift from references generally while")
print("                                            the row's own are comparatively retained")

  test_refs                          loaded  358
  test_gen__baseline                 loaded  488
  test_gen__lora_r8_step1000         loaded  488
  test_gen__lora_r8_step4000         loaded  488
  test_gen__qlora_r8_step2000        loaded  488
  test_gen__sp_iii_p139_step2000     loaded  488

arm                    space       own   others   margin     d own  d others  d margin   p margin
baseline               dino     0.4349   0.1784   0.2565         -         -         -          -
baseline               clip     0.7605   0.6483   0.1122         -         -         -          -
baseline               arc      0.3035   0.0642   0.2379         -         -         -          -

lora_r8_step1000       dino     0.4286   0.1792   0.2494   -0.0063   +0.0008   -0.0071   2.24e-08
lora_r8_step1000       clip     0.7599   0.6509   0.1090   -0.0006   +0.0026   -0.0032   8.09e-07
lora_r8_step1000       arc      0.2736   0.0677   0.2048   -0.0298   +0.0032   -0.0331   3.32e-22

lora_r8_step4000 